# Módulo 3 - Ejercicio 2 (Solución Completa)
## Algoritmo de Grover: Oráculo de Fase, Difusor y Muestreo

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

In [2]:
def construir_oraculo():
    qc = QuantumCircuit(3, name="Oráculo |110>")
    qc.x(0)
    qc.h(2)
    qc.mcx([0, 1], 2)
    qc.h(2)
    qc.x(0)
    return qc

oraculo = construir_oraculo()
print(oraculo.draw(output='text'))

     ┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     └───┘  │  └───┘
q_1: ───────■───────
     ┌───┐┌─┴─┐┌───┐
q_2: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘


In [3]:
def construir_difusor(n=3):
    qc = QuantumCircuit(n, name="Difusor")
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

difusor = construir_difusor(3)
print(difusor.draw(output='text'))

     ┌───┐┌───┐          ┌───┐┌───┐     
q_0: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐
q_2: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘└───┘└───┘└───┘└───┘


In [4]:
n = 3
N = 2**n
k_optimo = int(np.round(np.pi / 4 * np.sqrt(N)))

qc_grover = QuantumCircuit(n)
qc_grover.h(range(n))
qc_grover.barrier()

for _ in range(k_optimo):
    qc_grover.compose(oraculo, inplace=True)
    qc_grover.barrier()
    qc_grover.compose(difusor, inplace=True)
    qc_grover.barrier()

qc_grover.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc_grover], shots=2048)
counts = job.result()[0].data.meas.get_counts()

print("Conteos observados en Grover:", counts)
print(f"Probabilidad de éxito |110>: {counts.get('110', 0)/2048*100:.2f}%")

Conteos observados en Grover: {'110': 1946, '111': 14, '000': 16, '011': 17, '101': 14, '100': 13, '001': 12, '010': 16}
Probabilidad de éxito |110>: 95.02%
